In [1]:
# 导入所需的库
import pandas as pd
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from ipywidgets import interact, IntSlider

# 开启 Jupyter Notebook 的 Bokeh 输出
output_notebook()

Loading BokehJS ...

In [3]:
# 加载奥林匹克运动员数据集
dataset = pd.read_csv('olympia2016_athletes.csv')

# 对每个国家（nationality）进行数据聚合统计
countries_df = dataset.groupby('nationality', as_index=False).agg({
    'gold': 'sum',
    'silver': 'sum',
    'bronze': 'sum',
    'id': 'count'
}).rename(columns={'id': 'num_athletes'})

countries_df['total_medals'] = countries_df['gold'] + countries_df['silver'] + countries_df['bronze']

# 获取最大值，用于设置滑块边界
max_medals_val = int(countries_df['total_medals'].max())
max_athletes_val = int(countries_df['num_athletes'].max())

# 定义滑块控件：垂直方向的运动员人数滑块，水平方向的总奖牌数滑块
athletes_slider = IntSlider(min=0, max=max_athletes_val, value=max_athletes_val, orientation='vertical', description='Max. Athletes:')
medals_slider = IntSlider(min=0, max=max_medals_val, value=max_medals_val, orientation='horizontal', description='Max. Medals:')

# 定义绘制和过滤图表的核心函数 get_plot
def get_plot(max_athletes, max_medals):
    filtered_df = countries_df[
        (countries_df['num_athletes'] <= max_athletes) & 
        (countries_df['total_medals'] <= max_medals)
    ]
    
    source = ColumnDataSource(data=dict(
        nationality=filtered_df['nationality'],
        num_athletes=filtered_df['num_athletes'],
        total_medals=filtered_df['total_medals'],
        gold=filtered_df['gold'],
        silver=filtered_df['silver'],
        bronze=filtered_df['bronze']
    ))
    
    TOOLTIPS = [
        ("Country", "@nationality"),
        ("Athletes", "@num_athletes"),
        ("Medals", "@total_medals"),
        ("Gold", "@gold"),
        ("Silver", "@silver"),
        ("Bronze", "@bronze")
    ]
    
    plot = figure(title='Rio Olympics 2016 - Medal comparison',
                  x_axis_label='Number of Medals',
                  y_axis_label='Num of Athletes',
                  tooltips=TOOLTIPS,
                  width=700, height=500)
    
    plot.scatter('total_medals', 'num_athletes', size=15, source=source, alpha=0.6, color='navy')
    
    return plot

# 使用 @interact 绑定滑块与图表渲染
@interact(max_athletes=athletes_slider, max_medals=medals_slider)
def interactive_plot(max_athletes, max_medals):
    return show(get_plot(max_athletes, max_medals))

interactive(children=(IntSlider(value=567, description='Max. Athletes:', max=567, orientation='vertical'), Int…